# Start Implementation Notebook
Notebook này khởi tạo luồng triển khai tối thiểu có thể chạy được ngay: setup, cấu trúc dự án, model dữ liệu, service function, validation, unit tests và end-to-end run.

## 1. Set Up Python Environment and Dependencies
Phần này kiểm tra interpreter hiện tại, gợi ý lệnh cài thư viện, và import các thư viện cốt lõi.

In [1]:
import sys
import subprocess
from pathlib import Path

import pandas as pd

required_packages = ["pandas", "pytest"]

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Required packages:", required_packages)
print("If needed, run in terminal:")
print("  pip install pandas pytest")

Python executable: C:\Users\this pc\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe
Python version: 3.13.13
Required packages: ['pandas', 'pytest']
If needed, run in terminal:
  pip install pandas pytest


## 2. Create Project Structure in the Workspace
Tạo cấu trúc tối thiểu gồm thư mục mã nguồn, test và dữ liệu mẫu.

In [2]:
workspace_root = Path.cwd()
project_dirs = [
    workspace_root / "src",
    workspace_root / "tests",
    workspace_root / "data" / "sample",
]

for d in project_dirs:
    d.mkdir(parents=True, exist_ok=True)

print("Created/verified directories:")
for d in project_dirs:
    print("-", d)

print("\nWorkspace root:", workspace_root)

Created/verified directories:
- d:\Data impact\404_Not_Found\src
- d:\Data impact\404_Not_Found\tests
- d:\Data impact\404_Not_Found\data\sample

Workspace root: d:\Data impact\404_Not_Found


## 3. Define Configuration and Constants
Dùng dataclass để quản lý cấu hình chạy và đường dẫn đầu vào/đầu ra.

In [3]:
from dataclasses import dataclass
from typing import Literal


@dataclass(frozen=True)
class AppConfig:
    data_dir: Path
    output_dir: Path
    inactivity_days: int = 60
    risk_threshold: float = 0.65
    run_mode: Literal["dev", "prod"] = "dev"


config = AppConfig(
    data_dir=workspace_root / "data" / "sample",
    output_dir=workspace_root / "data" / "sample",
)
print(config)

AppConfig(data_dir=WindowsPath('d:/Data impact/404_Not_Found/data/sample'), output_dir=WindowsPath('d:/Data impact/404_Not_Found/data/sample'), inactivity_days=60, risk_threshold=0.65, run_mode='dev')


## 4. Implement Core Data Model
Định nghĩa data model cho một khách hàng và kiểm tra hiển thị dữ liệu có cấu trúc.

In [4]:
@dataclass(frozen=True)
class CustomerProfile:
    customer_id: int
    frequency: int
    monetary: float
    recency_days: int


sample_customer = CustomerProfile(customer_id=10001, frequency=6, monetary=580.5, recency_days=22)
print(sample_customer)
print(sample_customer.__dict__)

CustomerProfile(customer_id=10001, frequency=6, monetary=580.5, recency_days=22)
{'customer_id': 10001, 'frequency': 6, 'monetary': 580.5, 'recency_days': 22}


## 5. Implement First Service Function
Xây hàm tính risk score đơn giản từ RFM và kiểm thử nhanh bằng dữ liệu mẫu.

In [5]:
def compute_risk_score(profile: CustomerProfile) -> float:
    """Return risk score in [0, 1]. Higher means higher churn risk."""
    frequency_factor = max(0.0, 1.0 - min(profile.frequency, 12) / 12.0)
    recency_factor = min(profile.recency_days, 180) / 180.0
    monetary_factor = max(0.0, 1.0 - min(profile.monetary, 2000.0) / 2000.0)
    score = 0.4 * recency_factor + 0.4 * frequency_factor + 0.2 * monetary_factor
    return float(round(min(max(score, 0.0), 1.0), 4))

print("Risk score:", compute_risk_score(sample_customer))

Risk score: 0.3908


## 6. Add Input Validation and Error Handling
Bổ sung guard clauses để hàm fail-fast khi đầu vào không hợp lệ, sau đó chạy test dương/âm.

In [6]:
def validate_profile(profile: CustomerProfile) -> None:
    if profile.frequency < 0:
        raise ValueError("frequency must be >= 0")
    if profile.monetary < 0:
        raise ValueError("monetary must be >= 0")
    if profile.recency_days < 0:
        raise ValueError("recency_days must be >= 0")


def safe_compute_risk_score(profile: CustomerProfile) -> float:
    validate_profile(profile)
    return compute_risk_score(profile)

# Positive case
print("Validated risk score:", safe_compute_risk_score(sample_customer))

# Negative case
try:
    bad_customer = CustomerProfile(customer_id=10002, frequency=-1, monetary=100, recency_days=5)
    safe_compute_risk_score(bad_customer)
except ValueError as e:
    print("Expected validation error:", e)

Validated risk score: 0.3908
Expected validation error: frequency must be >= 0


## 7. Write Unit Tests for Core Logic
Viết test theo pytest-style cho case thường, case biên và case lỗi kỳ vọng.

In [7]:
src_file = workspace_root / "src" / "risk_scoring.py"
test_file = workspace_root / "tests" / "test_risk_scoring.py"

src_code = '''
from dataclasses import dataclass


@dataclass(frozen=True)
class CustomerProfile:
    customer_id: int
    frequency: int
    monetary: float
    recency_days: int


def compute_risk_score(profile: CustomerProfile) -> float:
    frequency_factor = max(0.0, 1.0 - min(profile.frequency, 12) / 12.0)
    recency_factor = min(profile.recency_days, 180) / 180.0
    monetary_factor = max(0.0, 1.0 - min(profile.monetary, 2000.0) / 2000.0)
    score = 0.4 * recency_factor + 0.4 * frequency_factor + 0.2 * monetary_factor
    return float(round(min(max(score, 0.0), 1.0), 4))


def validate_profile(profile: CustomerProfile) -> None:
    if profile.frequency < 0:
        raise ValueError("frequency must be >= 0")
    if profile.monetary < 0:
        raise ValueError("monetary must be >= 0")
    if profile.recency_days < 0:
        raise ValueError("recency_days must be >= 0")


def safe_compute_risk_score(profile: CustomerProfile) -> float:
    validate_profile(profile)
    return compute_risk_score(profile)
'''

test_code = '''
import pytest
from src.risk_scoring import CustomerProfile, safe_compute_risk_score


def test_compute_risk_score_normal_case():
    profile = CustomerProfile(customer_id=1, frequency=6, monetary=500.0, recency_days=30)
    score = safe_compute_risk_score(profile)
    assert 0.0 <= score <= 1.0


def test_compute_risk_score_edge_case_zero_activity():
    profile = CustomerProfile(customer_id=2, frequency=0, monetary=0.0, recency_days=180)
    score = safe_compute_risk_score(profile)
    assert score >= 0.8


def test_validation_failure_negative_frequency():
    profile = CustomerProfile(customer_id=3, frequency=-1, monetary=10.0, recency_days=5)
    with pytest.raises(ValueError):
        safe_compute_risk_score(profile)
'''

src_file.write_text(src_code, encoding="utf-8")
test_file.write_text(test_code, encoding="utf-8")

print("Source module written:", src_file)
print("Test file written:", test_file)
print(test_file.read_text(encoding="utf-8")[:300], "...")

Source module written: d:\Data impact\404_Not_Found\src\risk_scoring.py
Test file written: d:\Data impact\404_Not_Found\tests\test_risk_scoring.py

import pytest
from src.risk_scoring import CustomerProfile, safe_compute_risk_score


def test_compute_risk_score_normal_case():
    profile = CustomerProfile(customer_id=1, frequency=6, monetary=500.0, recency_days=30)
    score = safe_compute_risk_score(profile)
    assert 0.0 <= score <= 1.0


d ...


## 8. Run Tests and Execute a Minimal End-to-End Example
Chạy pytest và mô phỏng luồng từ danh sách khách hàng đầu vào đến bảng xếp hạng ưu tiên giữ chân.

In [8]:
# Run pytest in a subprocess so output resembles terminal behavior.
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(test_file)],
    cwd=str(workspace_root),
    capture_output=True,
    text=True,
)

print("Pytest exit code:", result.returncode)
print("--- stdout ---")
print(result.stdout if result.stdout else "<no stdout>")
print("--- stderr ---")
print(result.stderr if result.stderr else "<no stderr>")

if result.returncode != 0:
    raise RuntimeError("Pytest failed. Stop E2E flow until tests pass.")

# Minimal E2E flow
customers = [
    CustomerProfile(customer_id=101, frequency=2, monetary=120.0, recency_days=75),
    CustomerProfile(customer_id=102, frequency=9, monetary=1850.0, recency_days=12),
    CustomerProfile(customer_id=103, frequency=1, monetary=60.0, recency_days=140),
]

rows = []
for c in customers:
    rows.append({
        "customer_id": c.customer_id,
        "risk_score": safe_compute_risk_score(c),
    })

risk_df = pd.DataFrame(rows).sort_values("risk_score", ascending=False)
risk_df["priority"] = pd.cut(
    risk_df["risk_score"],
    bins=[-0.01, 0.35, 0.65, 1.01],
    labels=["low", "medium", "high"],
)

output_file = config.output_dir / "risk_ranking_sample.csv"
risk_df.to_csv(output_file, index=False)

print("\nTop at-risk customers:")
print(risk_df)
print("\nSaved:", output_file)

Pytest exit code: 0
--- stdout ---
...                                                                      [100%]
3 passed in 0.34s

--- stderr ---
<no stderr>

Top at-risk customers:
   customer_id  risk_score priority
2          103      0.8718     high
0          101      0.6880     high
1          102      0.1417      low

Saved: d:\Data impact\404_Not_Found\data\sample\risk_ranking_sample.csv
